In [ ]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
from pathlib import Path
from tqdm import tqdm
import numpy as np
from scipy import signal
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import pandas as pd
import cv2
from PIL import Image
import lovely_tensors as lt
lt.monkey_patch()
import imageio.v3 as iio
import torch

from hmr4d.utils.pylogger import Log
import hydra
from hydra import compose, initialize_config_module
from hydra.core.global_hydra import GlobalHydra
from hmr4d.configs import register_store_gvhmr
from hmr4d.model.gvhmr.gvhmr_pl_demo import DemoPL

from hmr4d.utils.preproc import Tracker, Extractor, VitPoseExtractor, SimpleVO
from hmr4d.utils.geo.hmr_cam import get_bbx_xys_from_xyxy, estimate_K
from hmr4d.utils.preproc.tracker import pick_main_walking_id, find_and_optionally_merge
from hmr4d.utils.preproc.vitpose_pytorch.src.vitpose_infer.pose_utils.pose_viz import joints_dict
from hmr4d.utils.geo_transform import compute_cam_angvel, move_to_start_point_face_z
from hmr4d.utils.body_model import BodyModelSMPLX
from hmr4d.utils.body_model.smplx_lite import SmplxLiteV437Coco17
from hmr4d.utils.vis.renderer import Renderer
from hmr4d.utils.vis.cv2_utils import draw_bbx_xyxy_on_image_batch, draw_kpts_with_conf_batch
from hmr4d.utils.video_io_utils import get_writer


In [ ]:
# GlobalHydra.instance().clear()
with initialize_config_module(version_base="1.3", config_module="hmr4d.configs"):
    register_store_gvhmr()
    cfg = compose(config_name="demo")
    
model = hydra.utils.instantiate(cfg.model, _recursive_=False)
model.load_pretrained_model(cfg.ckpt_path)
model = model.eval().cuda()

In [ ]:
smplx = BodyModelSMPLX(
    model_path="inputs/checkpoints/body_models", model_type="smplx",
    gender="neutral", num_pca_comps=12, flat_hand_mean=False,
).cuda()
smplx_coco = SmplxLiteV437Coco17().cuda()

tracker = Tracker()
vitpose_extractor = VitPoseExtractor()
extractor = Extractor()

In [ ]:
video_path = Path("/data/datasets/Chippy_medical/2025_10-11/1370_T2.mp4")
output_path = Path("/data/datasets/Chippy_medical/2025_10-11_results")
output_path.mkdir(exist_ok=True, parents=True)

length, height, width, c = iio.improps(video_path, plugin="pyav").shape
if length == 0:
    video = cv2.VideoCapture(str(video_path))
    fps = video.get(cv2.CAP_PROP_FPS)
    length = int(video.get(cv2.CAP_PROP_FRAME_COUNT))-1
print(f"{str(video_path)}: {width}x{height}x{length}")

final_results = torch.load(output_path / f"{video_path.stem}.pt")

In [ ]:
track_history = tracker.track(video_path)
vid_length = len(track_history)
id_to_frame_ids, id_to_bbx_xyxys, id_sorted = tracker.sort_track_length(track_history, video_path)

for k in id_sorted:
    lst = id_to_frame_ids[k]
    ranges = np.split(lst, np.where(np.diff(lst)!=1)[0]+1)
    ranges = [(x[0], x[-1]) if len(x) > 1 else (x[0], x[0]) for x in ranges]
    print(f"track id {k}: {len(lst)} frames, ranges: {ranges}")

In [ ]:
best_id, info = pick_main_walking_id(
    id_to_frame_ids=id_to_frame_ids,
    id_to_bbx_xyxys=id_to_bbx_xyxys,
    total_frames=len(track_history),   # or video length
    video_w=width,
    video_h=height,
)
print(best_id)
info

In [ ]:
track_ids = [4]
frame_ids, bbx_xyxys = [], []
for track_id in track_ids:
    tmp_id = torch.tensor(id_to_frame_ids[track_id])  # (N,)
    frame_ids.append(tmp_id)
    tmp_xyxy = torch.tensor(id_to_bbx_xyxys[track_id])  # (N, 4)
    bbx_xyxys.append(tmp_xyxy)
frame_ids = torch.cat(frame_ids)
bbx_xyxys = torch.cat(bbx_xyxys)

bbx_xyxy = tracker.interpolate_smooth_bbx(frame_ids, bbx_xyxys, length=vid_length)
bbx_xys = get_bbx_xys_from_xyxy(bbx_xyxy, base_enlarge=1.2).float()

In [ ]:
i = 300
img = iio.imread(video_path, index=i)  # (L, H, W, 3) RGB
video_overlay = draw_bbx_xyxy_on_image_batch(bbx_xyxy[i:i+1], [img], thickness=8)
Image.fromarray(video_overlay[0]).resize((width // 2, height // 2))

In [ ]:
vitpose = vitpose_extractor.extract(str(video_path), bbx_xys) # (L, 17, 3)
vit_features = extractor.extract_video_features(str(video_path), bbx_xys) # (L, 1024)

simple_vo = SimpleVO(video_path, scale=0.5, step=8, method="sift", f_mm=None)
vo_results = simple_vo.compute() # (L, 4, 4)
R_w2c = torch.from_numpy(vo_results[:, :3, :3]) # (L, 3, 3)
K_fullimg = estimate_K(width, height).repeat(length, 1, 1) # (L, 3, 3)

In [ ]:
data = {
    "length": torch.tensor(length),
    "bbx_xys": bbx_xys,
    "kp2d": vitpose,
    "K_fullimg": K_fullimg,
    "cam_angvel": compute_cam_angvel(R_w2c),
    "f_imgseq": vit_features,
}
pred = model.predict(data, static_cam=False)

final_results = {
    "input_video" : str(video_path.absolute()),
    'model_dir': str(cfg.ckpt_path),
    "dimensions" : (length, width, height),
    "bbx_xyxy": bbx_xyxy, "bbx_xys": bbx_xys,
    "vitpose": vitpose,
    'vit_features': vit_features,
    'R_w2c': R_w2c,
    'K_fullimg': K_fullimg,
    'smpl_params_global' : {k:v.cpu() for k,v in pred["smpl_params_global"].items()},
    'smpl_params_incam' : {k:v.cpu() for k,v in pred["smpl_params_incam"].items()},
}
torch.save(final_results, output_path / f"{video_path.stem}.pt")

In [ ]:
renderer_c = Renderer(width, height, device="cuda", faces=smplx.faces, K=estimate_K(width, height))

df = []
for frame in range(len(final_results['vitpose'])):
    part = final_results['vitpose'][frame]
    sub_df = [[frame, i, joints_dict()['coco']['keypoints'].get(i, 'None'), f"{row[0]:.6f}", f"{row[1]:.6f}", f"{row[2]:.4f}"] for i, row in enumerate(part)]
    df.extend(sub_df)
df = pd.DataFrame(df, columns=['frame', 'joint_idx', 'joint_name', 'x', 'y', 'confidence'])
df.to_csv(output_path / f"{video_path.stem}_vitpose.csv", index=False)

smplx_coco_camera_verts, smplx_coco_camera_joints = smplx_coco(**pred['smpl_params_incam'])
smplx_coco_camera_joints, smplx_coco_camera_valid = renderer_c.project_points_to_full_image(smplx_coco_camera_joints)

smplx_coco_global_verts, smplx_coco_global_joints  = smplx_coco(**pred["smpl_params_global"], return_all_verts=True)
smplx_coco_global_verts, smplx_coco_global_joints = move_to_start_point_face_z(
    smplx_coco_global_verts[..., :132, :], 
    smplx_coco.smplx2coco17_interestd.T,
    hip_j = [11, 12], 
    shoulder_j = [5, 6]
)
df = []
for frame in range(len(smplx_coco_camera_joints)):
    part = smplx_coco_camera_joints[frame]
    sub_df = [[frame, i, joints_dict()['coco']['keypoints'].get(i, 'None'), f"{row[0]:.6f}", f"{row[1]:.6f}"] for i, row in enumerate(part)]
    df.extend(sub_df)
df = pd.DataFrame(df, columns=['frame', 'joint_idx', 'joint_name', 'x', 'y'])
df.to_csv(output_path / f"{video_path.stem}_coco_camera_joints.csv", index=False)

df = []
for frame in range(len(smplx_coco_global_joints)):
    part = smplx_coco_global_joints[frame]
    sub_df = [[frame, i, joints_dict()['coco']['keypoints'].get(i, 'None'), f"{row[0]:.6f}", f"{row[1]:.6f}", f"{row[2]:.6f}"] for i, row in enumerate(part)]
    df.extend(sub_df)
df = pd.DataFrame(df, columns=['frame', 'joint_idx', 'joint_name', 'x', 'y', 'z'])
df.to_csv(output_path / f"{video_path.stem}_coco_global_joints.csv", index=False)

In [ ]:
# smpl
renderer_c = Renderer(width, height, device="cuda", faces=smplx.faces, K=estimate_K(width, height))

smplx_out = smplx(**{k: v.to('cuda') for k, v in final_results['smpl_params_incam'].items()})

i = 60
img_raw = cv2.resize(iio.imread(video_path, index=i), (width, height))
img_annot = draw_bbx_xyxy_on_image_batch(final_results['bbx_xyxy'][i:i+1], [img_raw], thickness=8)[0]
img_annot = draw_kpts_with_conf_batch(
    [img_annot[..., ::-1]], final_results['vitpose'][i:i+1, ..., :2], 
                            final_results['vitpose'][i:i+1, ...,  2], thickness=8)[0][...,::-1]

img_cam = renderer_c.render_mesh(smplx_out.vertices[i].cuda(), img_raw)
img_debug = Image.fromarray(np.concatenate([img_annot, img_cam], axis=1))
img_debug.save(output_path / f"{video_path.stem}_{i}_debug.jpg")
img_debug

In [ ]:
writer = get_writer(output_path / f"{video_path.stem}_debug.mp4", fps=10, crf=23)
for i in tqdm(range(0, length, 3), desc=f"Rendering Global"):
    img_raw = cv2.resize(iio.imread(video_path, index=i), (width, height))
    img_annot = draw_bbx_xyxy_on_image_batch(final_results['bbx_xyxy'][i:i+1], [img_raw], thickness=8)[0]
    img_annot = draw_kpts_with_conf_batch(
        [img_annot[..., ::-1]], final_results['vitpose'][i:i+1, ..., :2], 
                                final_results['vitpose'][i:i+1, ...,  2], thickness=8)[0][...,::-1]

    img_cam = renderer_c.render_mesh(smplx_out.vertices[i].cuda(), img_raw)
    img_debug = Image.fromarray(np.concatenate([img_annot, img_cam], axis=1)).resize((1920, 540))
    writer.write_frame(np.asarray(img_debug))
writer.close()

In [ ]:
# from hmr4d.utils.preproc.relpose.solver_two_view import TwoPairSolver, CameraParams, interpolate_missing_frames
# from hmr4d.utils.preproc.relpose.matcher_wrapper import Matcher
# from hmr4d.utils.preproc.relpose.utils import focal_length_from_mm
# from hmr4d.utils.video_io_utils import read_video_np

# frames = read_video_np(video_path, scale=0.5, rotate=0)
# F_all = frames.shape[0]
# sample_idxs = np.arange(0, F_all, 8)
# if sample_idxs[-1] != F_all - 1:
#     sample_idxs = np.concatenate([sample_idxs, [F_all - 1]])
# frames = frames[sample_idxs]
# F, H, W, C = frames.shape
# print(f"[SimpleVO] Choosen frames shape: {frames.shape}")

# matcher: Matcher = Matcher('sift')
# camera_params = CameraParams(W, H, focal_length=focal_length_from_mm(W, H, 24))
# solver = TwoPairSolver(camera_params, solver="pycolmap")

# T_w2c_list = [np.eye(4)]
# prev_frame = frames[0]
# for frame_idx in tqdm(range(1, len(frames))):
#     curr_frame = frames[frame_idx]
#     pts0, pts1 = matcher.match_np(prev_frame, curr_frame)
#     T_delta = solver.solve(pts0, pts1)  # T_delta = T_curr @ T_last^-1
#     T_w2c_list.append(T_delta @ T_w2c_list[-1])
#     prev_frame = curr_frame

# from PIL import ImageDraw

# point = [540, 280]
# tmp = Image.fromarray(video_overlay[0]).resize((960, 540))
# draw = ImageDraw.Draw(tmp)
# draw.ellipse((point[0]-10, point[1]-10, point[0]+10, point[1]+10), outline="red", width=5)
# tmp

## Gait Analysis: Step Length Detection

Detects foot strikes from the SMPLX global joint trajectories and computes per-step lengths (in cm).

**Assumptions**
- The subject stands up from a chair, walks straight toward the camera for ~3 m, turns around, and walks back to the chair.
- Global joints have been root-normalised by `move_to_start_point_face_z` so the subject starts at the origin facing +Z.
- Y axis = vertical (up); X axis = lateral; Z axis = forward walking direction.

In [ ]:
output_path = Path("/data/datasets/Chippy_medical/2025_08-09_results")
video_names = sorted([x.stem for x in output_path.glob("*.pt")])
video_names

In [ ]:
output_path = Path("/data/datasets/Chippy_medical/2025_08-09_results")
# video_names = sorted([x.stem for x in output_path.glob("*.pt")])
video_names = ['1395_T3']
fps = 30.0
min_dist   = max(5, int(fps * 0.25))   # minimum frames between strikes (~0.25 s at full speed)
prominence = 0.015                      # minimum height oscillation in metres (1.5 cm)

def get_traj(df, jidx):
    sub = df[df['joint_idx'] == jidx].sort_values('frame')
    return sub['frame'].values, sub[['x', 'y', 'z']].values.astype(float)

for video_name in video_names:
    df_global = pd.read_csv(output_path / f"{video_name}_coco_global_joints.csv")
    for col in ['x', 'y', 'z']:
        df_global[col] = df_global[col].astype(float)
        
    frames_arr, la = get_traj(df_global, 15)   # left  ankle
    _,           ra = get_traj(df_global, 16)   # right ankle
    _,           lh = get_traj(df_global, 11)   # left  hip
    _,           rh = get_traj(df_global, 12)   # right hip
    hip_center = (lh + rh) / 2 
    
    sigma = max(3, int(fps * 0.08))          # ~2-3 frames at 30 fps
    la_y_s   = gaussian_filter1d(la[:, 1],           sigma)
    ra_y_s   = gaussian_filter1d(ra[:, 1],           sigma)
    hip_z_s  = gaussian_filter1d(hip_center[:, 2],   sigma)

    l_strikes, _ = signal.find_peaks(-la_y_s, distance=min_dist, prominence=prominence)
    r_strikes, _ = signal.find_peaks(-ra_y_s, distance=min_dist, prominence=prominence)

    turn_idx   = int(np.argmax(hip_z_s))
    turn_frame = frames_arr[turn_idx]
    
    all_strikes = (
        [{'fi': fi, 'frame': int(frames_arr[fi]), 'foot': 'L', 'pos': la[fi]} for fi in l_strikes] +
        [{'fi': fi, 'frame': int(frames_arr[fi]), 'foot': 'R', 'pos': ra[fi]} for fi in r_strikes]
    )
    all_strikes.sort(key=lambda s: s['frame'])

    rows = []
    for i in range(1, len(all_strikes)):
        a, b = all_strikes[i - 1], all_strikes[i]
        if a['foot'] == b['foot']:
            continue                           # same foot twice = skip
        dp    = b['pos'] - a['pos']
        mid   = (a['frame'] + b['frame']) / 2
        phase = 'walk_out' if mid <= turn_frame else 'walk_back'
        rows.append({
            'from_frame':      a['frame'],
            'to_frame':        b['frame'],
            'from_foot':       a['foot'],
            'to_foot':         b['foot'],
            'step_length_cm':  round(abs(dp[2]) * 100, 1),   # |ΔZ|
            'step_width_cm':   round(abs(dp[0]) * 100, 1),   # |ΔX|
            'phase':           phase,
        })

    df_steps = pd.DataFrame(rows)
    out_csv = output_path / f"{video_name}_step_lengths.csv"
    df_steps.to_csv(out_csv, index=False)
    print(f"\nSaved → {out_csv}")

In [ ]:
# --- Detect turn-around frame via peak hip-Z ---
turn_idx   = int(np.argmax(hip_z_s))
turn_frame = frames_arr[turn_idx]
print(f"Turn-around detected at frame {turn_frame}  (hip Z = {hip_z_s[turn_idx]:.3f} m)")

# --- Combine all strikes, sorted by frame ---
all_strikes = (
    [{'fi': fi, 'frame': int(frames_arr[fi]), 'foot': 'L', 'pos': la[fi]} for fi in l_strikes] +
    [{'fi': fi, 'frame': int(frames_arr[fi]), 'foot': 'R', 'pos': ra[fi]} for fi in r_strikes]
)
all_strikes.sort(key=lambda s: s['frame'])

# --- Compute step length and step width for consecutive ALTERNATING foot strikes ---
#
#  Coordinate system after move_to_start_point_face_z:
#    X = lateral (side-to-side)
#    Y = vertical (up)
#    Z = forward  (walking direction)
#
#  For a displacement vector dp = (dX, dY, dZ) between two consecutive heel strikes:
#    step_length = |dZ|   — forward progress, regardless of walk direction
#    step_width  = |dX|   — mediolateral separation between feet
#
#  Note: the previous code used sqrt(dX²+dZ²) which conflates the two.
rows = []
for i in range(1, len(all_strikes)):
    a, b = all_strikes[i - 1], all_strikes[i]
    if a['foot'] == b['foot']:
        continue                           # same foot twice = skip
    dp    = b['pos'] - a['pos']
    mid   = (a['frame'] + b['frame']) / 2
    phase = 'walk_out' if mid <= turn_frame else 'walk_back'
    rows.append({
        'from_frame':      a['frame'],
        'to_frame':        b['frame'],
        'from_foot':       a['foot'],
        'to_foot':         b['foot'],
        'step_length_cm':  round(abs(dp[2]) * 100, 1),   # |ΔZ|
        'step_width_cm':   round(abs(dp[0]) * 100, 1),   # |ΔX|
        'phase':           phase,
    })

df_steps = pd.DataFrame(rows)
print("\nAll detected steps:")
print(df_steps.to_string(index=False))

# --- Save ---
out_csv = output_path / f"{video_path.stem}_step_lengths.csv"
df_steps.to_csv(out_csv, index=False)
print(f"\nSaved → {out_csv}")

In [ ]:
# --- Load precomputed global joints CSV ---
# (produced by the export cell above, or a previous run of this notebook)
df_global = pd.read_csv(output_path / f"{video_path.stem}_coco_global_joints.csv")
for col in ['x', 'y', 'z']:
    df_global[col] = df_global[col].astype(float)

# --- Get video FPS ---
_cap = cv2.VideoCapture(str(video_path))
fps = _cap.get(cv2.CAP_PROP_FPS) or 30.0
_cap.release()
print(f"FPS: {fps}   |   Frames with global joints: {df_global['frame'].nunique()}")

# --- Helper: extract (frames, xyz) for one COCO-17 joint index ---
def get_traj(df, jidx):
    sub = df[df['joint_idx'] == jidx].sort_values('frame')
    return sub['frame'].values, sub[['x', 'y', 'z']].values.astype(float)

# COCO-17 indices:  11=left_hip  12=right_hip  15=left_ankle  16=right_ankle
frames_arr, la = get_traj(df_global, 15)   # left  ankle
_,           ra = get_traj(df_global, 16)   # right ankle
_,           lh = get_traj(df_global, 11)   # left  hip
_,           rh = get_traj(df_global, 12)   # right hip

hip_center = (lh + rh) / 2   # proxy for centre-of-mass forward position

In [ ]:
# --- Smooth trajectories ---
sigma = max(3, int(fps * 0.08))          # ~2-3 frames at 30 fps
la_y_s   = gaussian_filter1d(la[:, 1],           sigma)
ra_y_s   = gaussian_filter1d(ra[:, 1],           sigma)
hip_z_s  = gaussian_filter1d(hip_center[:, 2],   sigma)

# --- Visualise ankle heights and hip forward position ---
fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

axes[0].plot(frames_arr, la[:, 1],  alpha=0.2, color='blue')
axes[0].plot(frames_arr, la_y_s,    lw=2, color='blue', label='Left ankle (smooth)')
axes[0].plot(frames_arr, ra[:, 1],  alpha=0.2, color='red')
axes[0].plot(frames_arr, ra_y_s,    lw=2, color='red',  label='Right ankle (smooth)')
axes[0].set_ylabel('Height Y (m)')
axes[0].set_title('Ankle Heights in Global Space  (Y = vertical)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(frames_arr, hip_center[:, 2], alpha=0.2, color='green')
axes[1].plot(frames_arr, hip_z_s,          lw=2, color='green', label='Hip centre Z (smooth)')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Forward Z (m)')
axes[1].set_title('Hip Centre Forward Position  (walk direction = +Z)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Foot strike detection ---
# A foot strike = local minimum of ankle height (ankle lowest point during ground contact).
# Parameters: tune `prominence` if too many / too few strikes are detected.
min_dist   = max(5, int(fps * 0.25))   # minimum frames between strikes (~0.25 s at full speed)
prominence = 0.015                      # minimum height oscillation in metres (1.5 cm)

l_strikes, _ = signal.find_peaks(-la_y_s, distance=min_dist, prominence=prominence)
r_strikes, _ = signal.find_peaks(-ra_y_s, distance=min_dist, prominence=prominence)

print(f"Left  foot strikes: {len(l_strikes):2d}  at frames {frames_arr[l_strikes].tolist()}")
print(f"Right foot strikes: {len(r_strikes):2d}  at frames {frames_arr[r_strikes].tolist()}")

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(frames_arr, la_y_s, lw=2, color='blue', label='Left ankle')
ax.plot(frames_arr, ra_y_s, lw=2, color='red',  label='Right ankle')
ax.scatter(frames_arr[l_strikes], la_y_s[l_strikes],
           color='blue', s=120, zorder=5, marker='v', label='Left strikes')
ax.scatter(frames_arr[r_strikes], ra_y_s[r_strikes],
           color='red',  s=120, zorder=5, marker='v', label='Right strikes')
ax.set_xlabel('Frame'); ax.set_ylabel('Ankle Height (m)')
ax.set_title('Detected Foot Strikes (▼)  —  adjust `prominence` or `min_dist` if needed')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- Detect turn-around frame via peak hip-Z ---
turn_idx   = int(np.argmax(hip_z_s))
turn_frame = frames_arr[turn_idx]
print(f"Turn-around detected at frame {turn_frame}  (hip Z = {hip_z_s[turn_idx]:.3f} m)")

# --- Combine all strikes, sorted by frame ---
all_strikes = (
    [{'fi': fi, 'frame': int(frames_arr[fi]), 'foot': 'L', 'pos': la[fi]} for fi in l_strikes] +
    [{'fi': fi, 'frame': int(frames_arr[fi]), 'foot': 'R', 'pos': ra[fi]} for fi in r_strikes]
)
all_strikes.sort(key=lambda s: s['frame'])

# --- Compute step length and step width for consecutive ALTERNATING foot strikes ---
#
#  Coordinate system after move_to_start_point_face_z:
#    X = lateral (side-to-side)
#    Y = vertical (up)
#    Z = forward  (walking direction)
#
#  For a displacement vector dp = (dX, dY, dZ) between two consecutive heel strikes:
#    step_length = |dZ|   — forward progress, regardless of walk direction
#    step_width  = |dX|   — mediolateral separation between feet
#
#  Note: the previous code used sqrt(dX²+dZ²) which conflates the two.
rows = []
for i in range(1, len(all_strikes)):
    a, b = all_strikes[i - 1], all_strikes[i]
    if a['foot'] == b['foot']:
        continue                           # same foot twice = skip
    dp    = b['pos'] - a['pos']
    mid   = (a['frame'] + b['frame']) / 2
    phase = 'walk_out' if mid <= turn_frame else 'walk_back'
    rows.append({
        'from_frame':      a['frame'],
        'to_frame':        b['frame'],
        'from_foot':       a['foot'],
        'to_foot':         b['foot'],
        'step_length_cm':  round(abs(dp[2]) * 100, 1),   # |ΔZ|
        'step_width_cm':   round(abs(dp[0]) * 100, 1),   # |ΔX|
        'phase':           phase,
    })

df_steps = pd.DataFrame(rows)
print("\nAll detected steps:")
print(df_steps.to_string(index=False))

# --- Save ---
out_csv = output_path / f"{video_path.stem}_step_lengths.csv"
df_steps.to_csv(out_csv, index=False)
print(f"\nSaved → {out_csv}")

In [ ]:
# --- Summary statistics: step length and step width ---
print("=" * 70)
print("  GAIT SUMMARY")
print("=" * 70)
for phase, label in [
    ('walk_out',  'Walk OUT  (toward camera)'),
    ('walk_back', 'Walk BACK (to chair)'),
    (None,        'Overall'),
]:
    sub = df_steps if phase is None else df_steps[df_steps['phase'] == phase]
    if len(sub) == 0:
        print(f"\n{label}: no steps detected"); continue

    print(f"\n{label}  (n={len(sub)}):")
    for col, name in [('step_length_cm', 'Length |ΔZ|'), ('step_width_cm', 'Width  |ΔX|')]:
        v = sub[col]
        print(f"  {name}:  mean={v.mean():.1f}  std={v.std():.1f}  "
              f"min={v.min():.1f}  max={v.max():.1f} cm")
    print(f"  Length steps (cm): {sub['step_length_cm'].tolist()}")
    print(f"  Width  steps (cm): {sub['step_width_cm'].tolist()}")

In [ ]:
# --- Top-down view: foot placements and step arrows ---
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(7, 10))

phase_colors = {'walk_out': 'steelblue', 'walk_back': 'darkorange'}

# Draw step arrows
for _, row in df_steps.iterrows():
    src = next(s for s in all_strikes if s['frame'] == row['from_frame'])
    dst = next(s for s in all_strikes if s['frame'] == row['to_frame'])
    ax.annotate(
        '', xy=(dst['pos'][0], dst['pos'][2]),
        xytext=(src['pos'][0], src['pos'][2]),
        arrowprops=dict(arrowstyle='->', color=phase_colors[row['phase']], lw=1.8),
    )
    # Label the step length at the midpoint
    mx = (src['pos'][0] + dst['pos'][0]) / 2
    mz = (src['pos'][2] + dst['pos'][2]) / 2
    ax.text(mx + 0.02, mz, f"{row['step_length_cm']:.0f} cm", fontsize=8, color='dimgray')

# Draw foot positions
for s in all_strikes:
    color, marker = ('blue', 'o') if s['foot'] == 'L' else ('red', 's')
    ax.scatter(s['pos'][0], s['pos'][2], color=color, marker=marker, s=110, zorder=5)
    ax.annotate(str(s['frame']), (s['pos'][0], s['pos'][2]),
                xytext=(0, 5), textcoords='offset points',
                fontsize=7, ha='center', color=color)

# Mark turn point
ax.axhline(hip_z_s[turn_idx], ls='--', color='gray', lw=1.2,
           label=f'Turn point  Z={hip_z_s[turn_idx]:.2f} m')

legend_elems = [
    Line2D([0],[0], marker='o', color='blue', ls='None', markersize=9, label='Left foot'),
    Line2D([0],[0], marker='s', color='red',  ls='None', markersize=9, label='Right foot'),
    Line2D([0],[0], color='steelblue',  lw=2, label='Walk out'),
    Line2D([0],[0], color='darkorange', lw=2, label='Walk back'),
    Line2D([0],[0], ls='--', color='gray', lw=1.2, label='Turn point'),
]
ax.legend(handles=legend_elems, loc='upper right')
ax.set_xlabel('X (m, lateral)')
ax.set_ylabel('Z (m, forward)')
ax.set_title('Foot Placements – Top-Down View\n(frame numbers shown at each strike)')
ax.set_aspect('equal')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()